# 0. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import f1_score, accuracy_score

from wsd.load_data import load_data

from linpub.metrics import accuracy

from lineval.disambiguation_eval_backup import ClusterByMeaningModelv3
from lineval.disambiguation_eval_backup import OpenAiWordSenseComparatorv3, show_plots
from wsd.models import DummyComparator, ClusterByMeaningModel

from etl.models import Candidate

# 1. 1st try (raw, model, dataset)

In [ ]:
X,y = load_data("fr")
for x in X :
    x.context = x.example
size = 400
X, y = X[:size], y[:size]

In [ ]:
openai_api_key = os.getenv('OPEN_AI_API_KEY')
comp = OpenAiWordSenseComparatorv3(openai_api_key,
                                   openai_model='gpt-4o',
                                   thought_process=True)
model1 = ClusterByMeaningModelv3(word_sense_comparator=comp)

In [ ]:
dummy2 = DummyComparator(probability=1)
model2 = ClusterByMeaningModel(comparator=dummy2)

dummy3 = DummyComparator(probability=0)
model3 = ClusterByMeaningModel(comparator=dummy3)

In [ ]:
#X, y_pred1 = model1.predict(X, verbose=True)
y_pred2 = model2.predict(X, verbose=True)
y_pred3 = model3.predict(X, verbose=True)

In [ ]:
accuracy(y_pred1, y),accuracy(y_pred2, y), accuracy(y_pred3, y)

In [ ]:
X[2].process

In [ ]:
df = pd.DataFrame(X)
df = df.drop(columns=["context"])
df["pred"] = [c.pred for c in X]
df["process"] = [c.process for c in X]
df['y_pred'] = y_pred1
df['y'] = y
df = df.sort_values(by=["lemma"])
df = df[df["process"] != "lone candidate"]
#reindex
df = df.reset_index(drop=True)
df["same_y"] = df["y"] == df["y"].shift(-1)
df["same_pred"] = df["y_pred"] == df["y_pred"].shift(-1)
df["correct"] = df["same_y"] == df["same_pred"]
df

In [ ]:
from lineval.disambiguation_eval_backup import show_plots
show_plots(df)

In [ ]:
df = df[(df["correct"] == False) | (df["correct"].shift(1) == False)]
df

In [ ]:

df.to_csv("test.csv", index=False)

# 2. 2nd try : massaged dataset, new metric

## 2.1 load

In [ ]:
X,y = load_data("fr")
for x in X :
    x.context = x.example
len(X), len(y)

In [ ]:
X_df = pd.DataFrame(X)
X_df = df.drop(columns=["example"])
X_df["context"] = [c.context for c in X]
X_df["y"] = y

## 2.1.2 Exploration

In [ ]:
# group by lemma and pos
cond = X_df.groupby(["lemma", "pos"]).size() <= 20# show only lemmas with more than 10 examples
X_df_20m = X_df.set_index(["lemma", "pos"]).loc[cond].reset_index()
X_df_20m.groupby(["lemma", "pos"]).size(). hist(bins=20)
plt.title("Number of examples in pos/lemma groups")

In [ ]:
# group by lemma and pos
cond = X_df.groupby(["lemma", "pos"]).size() > 20# show only lemmas with more than 10 examples
X_df_20p = X_df.set_index(["lemma", "pos"]).loc[cond].reset_index()
X_df_20p.groupby(["lemma", "pos"]).size().hist(bins=1020)
plt.title("Number of examples in pos/lemma groups")

In [ ]:
X_df.groupby(["lemma", "pos"]).size().sort_values(ascending=False).head(10)

In [ ]:
X_df[X_df["lemma"] == "dire"]["y"].value_counts()

In [ ]:
lone_cands = len(X_df.groupby(["lemma", "pos"]).filter(lambda x: len(x) == 1))
lone_cands, lone_cands/len(X_df)

## 2.2 Dire (say) vs dire (tell)

In [ ]:
X_df[(X_df["lemma"] == "dire") & (X_df["y"] == "bn:00093287v\n")].head(20) #say

In [ ]:
X_df[(X_df["lemma"] == "dire") & (X_df["y"] != "bn:00093287v\n")].head(20) #tell

## 2.3 Creating a balanded set

In [ ]:
# X_df_len3 = X_df.groupby(["lemma", "pos"]).filter(lambda x: len(x) == 3)
# X_df_bl_kinda = pd.DataFrame()
# for name, group in X_df_len3.groupby(["lemma", "pos"]):
#     if group["y"].nunique() == 2:
#         X_df_bl_kinda = pd.concat([X_df_bl_kinda, group])
# len(X_df_bl_kinda)

### Kind of a balanced dataset
-----

The set will be balanced, but the current model will have different predictions based on the order of elements.

#### Example:

#### Sequence: `meaning_a / meaning_b / meaning_a` (`a / b / a`)

##### Predictions:
1. **Sequence:** `a / b / a`
   - `a / a` (ref)
   - `b / a` (0): No more meaning to check -> New meaning created
   - `a / a` (1): Matched!
   - **API Calls:** 2

2. **Sequence:** `b / a / a`
   - `b / b` (ref)
   - `a / b` (0): No more meaning to check -> New meaning created
   - `a / b` (0)
   - `a / a` (1): Matched!
   - **API Calls:** 3

3. **Sequence:** `a / a / b`
   - `a / a` (ref)
   - `a / a` (1): Matched!
   - `a / b` (0)
   - `a / b` (0)
   - **API Calls:** 3

For full balance, control the order: `a / b / a`

-----

In [ ]:
def get_balanced_df(X,y):
    X_df = pd.DataFrame(X)
    X_df["context"] = [c.example for c in X]
    X_df = X_df.drop(columns=["example"])
    X_df["y"] = y
    X_df_len3 = X_df.groupby(["lemma", "pos"]).filter(lambda x: len(x) == 3)
    X_df_bl = pd.DataFrame()
    for _, group in X_df_len3.groupby(["lemma", "pos"]):
        if group["y"].nunique() == 2:
            #sort group by number of similar meanings
            group["count"] = group["y"].map(group["y"].value_counts())
            group = group.sort_values(by="count")
            # order will now be b/a/a or a/b/b so  we just permute the first two
            group = pd.concat([group.iloc[[1]], group.iloc[[0]], group.iloc[[2]]])
            X_df_bl = pd.concat([X_df_bl, group])
    X_df_bl = X_df_bl.reset_index(drop=True)
    return X_df_bl
X,y = load_data("fr")
X_df_bl = get_balanced_df(X,y)

## 2.4 Running

In [ ]:
def get_balanced_X_y(X_df_bl):
    bl_y = X_df_bl["y"].to_list()
    cand_df = X_df_bl.drop(columns=["y", "count"])
    bl_X = [Candidate(**row) for row in cand_df.to_dict(orient="records")]
    return bl_X, bl_y
bl_X, bl_y = get_balanced_x_y(X_df_bl)
len(bl_X), len(bl_y)

In [ ]:
len(bl_X), len(bl_y)

In [ ]:
#if there is a csv file with the results, load it
if os.path.exists("results.csv"):
    df = pd.read_csv("results.csv")
    y_pred1 = df["y_pred"].to_list()
    df.drop(columns=["y_pred"], inplace=True)
    X = [Candidate(**row) for row in df.to_dict(orient="records")]
else:
    X, y_pred1 = model1.predict(bl_X, verbose=True)

y_pred2 = model2.predict(bl_X, verbose=True)
y_pred3 = model3.predict(bl_X, verbose=True)

In [ ]:
df = pd.DataFrame(bl_X)
df["y_pred"] = y_pred1
df.to_csv("results.csv", index=False)

In [ ]:
accuracy(y_pred1, bl_y),accuracy(y_pred2, bl_y), accuracy(y_pred3, bl_y)

In [ ]:
bl_y[0], y_pred1[0], y_pred2[0], y_pred3[0]

In [ ]:
# adjusting y and y_pred for f1 score
X_df = pd.DataFrame(bl_X)
X_df_bl["y"] = bl_y
X_df_bl["y_pred"] = y_pred1
f1_y = []
f1_y_pred = []
for _, group in X_df_bl.groupby(["lemma", "pos"]):
    pred1 = group["y_pred"].iloc[1] == group["y_pred"].iloc[0]
    pred2 = group["y_pred"].iloc[2] == group["y_pred"].iloc[0]
    group["y"] = pd.Series(["ref", False, True])
    group["y_pred"] = pd.Series(["ref", pred1, pred2])
    f1_y.extend([False, True])
    f1_y_pred.extend([pred1, pred2])
len(f1_y), len(f1_y_pred), 210*2/3

In [ ]:
def get_f1_score(bl_X, bl_y, y_pred):
    X_df = pd.DataFrame(bl_X)
    X_df["y"] = bl_y
    X_df["y_pred"] = y_pred
    f1_y = []
    f1_y_pred = []
    for _, group in X_df.groupby(["lemma", "pos"]):
        pred1 = bool(group["y_pred"].iloc[1] == group["y_pred"].iloc[0])
        pred2 = bool(group["y_pred"].iloc[2] == group["y_pred"].iloc[0])
        # group["y"] = pd.Series(["ref", False, True])
        # group["y_pred"] = pd.Series(["ref", pred1, pred2])
        f1_y.extend([False, True])
        f1_y_pred.extend([pred1, pred2])
    return f1_score(f1_y, f1_y_pred) , accuracy_score(f1_y, f1_y_pred)

In [ ]:
get_f1_score(bl_X, bl_y, y_pred1), get_f1_score(bl_X, bl_y, y_pred2), get_f1_score(bl_X, bl_y, y_pred3)
print(f"--f1 scores--")
print(f"Model: {get_f1_score(bl_X, bl_y, y_pred1)[0]}")
print(f"Dummy(1): {get_f1_score(bl_X, bl_y, y_pred2)[0]}")
print(f"Dummy(0): {get_f1_score(bl_X, bl_y, y_pred3)[0]}")
print(f"--sklearn accuracy scores--")
print(f"Model: {get_f1_score(bl_X, bl_y, y_pred1)[1]}")
print(f"Dummy(1): {get_f1_score(bl_X, bl_y, y_pred2)[0]}")
print(f"Dummy(0): {get_f1_score(bl_X, bl_y, y_pred3)[1]}")
print(f"--linpub accuracy scores--")
print(f"Model: {accuracy(y_pred1, bl_y)}")
print(f"Dummy(1): {accuracy(y_pred2, bl_y)}")
print(f"Dummy(0): {accuracy(y_pred3, bl_y)}")

# plotting

In [ ]:
plot_df = pd.DataFrame(bl_X)
plot_df["pred"] = [c.pred for c in bl_X]
plot_df["process"] = [c.process for c in bl_X]
plot_df["y_pred"] = y_pred1
plot_df["y"] = bl_y
plot_df.drop(columns=["text_meaning", "text_romaji", "text_hiragana", "lemma_romaji", "lemma_hiragana", "example", "annotation","document", "status", ], inplace=True)
correct_df = pd.DataFrame()
for _, gp in plot_df.groupby(["lemma", "pos"]):
    pred1 = gp["y_pred"].iloc[1] != gp["y_pred"].iloc[0]
    pred2 = gp["y_pred"].iloc[2] == gp["y_pred"].iloc[0]
    gp = gp.copy()  # To avoid SettingWithCopyWarning
    gp["correct"] = pd.Series(["ref", pred1, pred2], index=gp.index)
    correct_df = pd.concat([correct_df, gp])
correct_df.head(20)

In [ ]:
correct_df[correct_df["correct"] == False]["pred"].value_counts(), correct_df[correct_df["correct"] == True]["pred"].value_counts()

In [ ]:
show_plots(correct_df)

# Conclusions


-------
1. base mdel and data

1.1 The dummy 1 is better that the model at 91% vs 90%

1.2 The dataset is unbalanced

1.3 The model errors seem to be due to errors in the y

-------
2. balancing the dataset

2.1. There are 1433 lone candidates, ~6% of the dataset

2.2 The top pos/lemma group is dire, with 1020 rows (2 different meanings)(944of the main maining, and then 74 of the 2nd) (say vs tell)

2.3 From the pos/lemma groups with 3 rows and 2 meanings, (210 rows) I made a balanced, ordered dataset where the model will have to do 2 outputs: 0 then 1 

2.4. The model is barely better than the baseline